In [1]:
import random
import numpy as np
import pickle
import json
import nltk
from nltk.stem import WordNetLemmatizer 
from tensorflow.keras.models import load_model

In [2]:
lemmatizer = WordNetLemmatizer()
intents = json.loads(open('questionAndAns.json').read())

In [3]:
words = pickle.load(open('words.pkl','rb'))
classes = pickle.load(open('classes.pkl','rb'))
model = load_model('chatbot_model.h5')

In [6]:
def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    # Added .lower() to match your training logic
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words] 
    return sentence_words

def bag_of_words(sentence):
    sentence_words = clean_up_sentence(sentence)
    bag = [0] * len(words)
    for w in sentence_words:
        for i, word in enumerate(words):
            if word == w:
                bag[i] = 1
    return np.array(bag) # <--- MOVE THIS OUTSIDE ALL LOOPS
def predict_class(sentence):
    bow = bag_of_words(sentence)
    # Added verbose=0 to hide the training-style progress bar
    res = model.predict(np.array([bow]), verbose=0)[0] 
    
    ERROR_THRESOLD = 0.5
    result = [[i,r] for i,r in enumerate(res) if r> ERROR_THRESOLD]
    result.sort(key = lambda x : x[1],reverse = True)
    return_list = []
    for r in result:
        return_list.append({'intent' : classes[r[0]],'probability': str(r[1]) })
    return return_list

def get_response(intent_list, intents_json):
    tag = intent_list[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if i['tag'] == tag:
            result = random.choice(i['responses'])
            return result # THIS MUST BE INSIDE THE IF-BLOCK
    return "I don't understand." # Fallback if no tag matches
print ("Perfeca")
while True:
    message = input("You: ")
    if message.lower() in ['exit', 'bye', 'see you', 'quit']:        
        print("Bot: Goodbye!")
        break
    ints = predict_class(message)
    if len(ints) > 0:
        res = get_response(ints, intents)
        print("Bot:", res)
    else:
        print("Bot: I'm not sure I understand. Could you rephrase that?")

Perfeca


You:  exit 


Bot: Goodbye! Take care of your health.


You:  exit


Bot: Goodbye!
